# Dashboard de Tendências Anuais (UFPB)

Este notebook carrega os relatórios de métricas de **múltiplos anos** (ex: 2023, 2024, ...) e gera gráficos de tendência para comparar o desempenho ao longo do tempo.

## 1. Importações e Configurações

In [1]:
import pandas as pd
import numpy as np
import altair as alt
import glob
import os
import re

# --- Configurações ---
# orgao = 'UFPB'
orgao = 'UFCG'
# Pasta base onde as subpastas anuais estão localizadas
base_dir = 'dadosViagens'
# Padrão do nome do arquivo de relatório
report_pattern_base = f"relatorio_metricas_scores_{orgao}_" # O resto será preenchido com ano e timestamp

# --- Função de Limpeza (Copiada do notebook anterior) ---
def clean_numeric_value(value_in):
    """ Tenta converter uma string (com unidades, R$, %, etc.) para float. """
    if pd.isna(value_in): return np.nan
    if isinstance(value_in, (int, float)): return float(value_in)
    try:
        s = str(value_in).strip()
        # Remove unidades comuns e R$ (ignorando case), mantém % para tratar depois
        s = re.sub(r'(R\$|\s?KgCO2eq|\s?Km)', '', s, flags=re.IGNORECASE).strip()
        is_percent = '%' in s
        s = s.replace('%', '').strip()
        # Trata separador de milhar PT-BR '.' e decimal ','
        if '.' in s and ',' in s and s.rfind('.') < s.rfind(','):
             s = s.replace('.', '').replace(',', '.')
        # Trata separador de milhar US/EN ',' (remove vírgulas)
        elif ',' in s:
             s = s.replace(',', '')
        num = float(s)
        # Se o valor original era percentual e o número é > 1, assume que é 0-100 e divide
        # (Ajuste se seus percentuais já são 0-1)
        # if is_percent and num > 1: num /= 100.0
        return num
    except: 
        return np.nan

## 2. Carregar e Consolidar Relatórios Anuais

In [2]:
all_dfs = []
reports_by_year = {}

# 1. Encontrar todos os arquivos de relatório em todas as subpastas anuais
search_pattern = os.path.join(base_dir, 'dados_viagens*', f"{report_pattern_base}*.csv")
all_report_files = glob.glob(search_pattern)
print(f"🔄 Encontrados {len(all_report_files)} arquivos de relatório correspondentes ao padrão...")

# 2. Extrair o ano e agrupar os arquivos por ano
year_regex = re.compile(f"{report_pattern_base}(\d{{4}})_.*\.csv") # Regex para capturar o ano (4 dígitos)
for f_path in all_report_files:
    match = year_regex.search(os.path.basename(f_path))
    if match:
        year = match.group(1) # Pega o ano capturado (ex: '2023')
        if year not in reports_by_year:
            reports_by_year[year] = []
        reports_by_year[year].append(f_path)

print(f"✅ Arquivos agrupados por ano: {list(reports_by_year.keys())}")

# 3. Carregar o arquivo MAIS RECENTE de cada ano e concatenar
for year, files in reports_by_year.items():
    if files:
        latest_file = max(files, key=os.path.getctime) # Encontra o mais recente pela data de modificação
        print(f"   - Carregando dados do ano {year} de: '{os.path.basename(latest_file)}'...")
        try:
            df_year = pd.read_csv(latest_file, sep=';', decimal=',')
            df_year['Ano'] = int(year) # Adiciona a coluna 'Ano'
            all_dfs.append(df_year)
        except Exception as e_load:
            print(f"     ⚠️ Erro ao carregar o arquivo '{latest_file}': {e_load}")

# 4. Criar DataFrame consolidado
if all_dfs:
    df_full_report = pd.concat(all_dfs, ignore_index=True)
    print(f"\n✅ Relatório anual consolidado criado com {len(df_full_report)} linhas.")
    print(df_full_report.head())
else:
    print("\n❌ Nenhum dado de relatório anual foi carregado. Não é possível continuar.")
    df_full_report = pd.DataFrame() # Cria DF vazio

🔄 Encontrados 3 arquivos de relatório correspondentes ao padrão...
✅ Arquivos agrupados por ano: ['2023', '2024']
   - Carregando dados do ano 2023 de: 'relatorio_metricas_scores_UFCG_2023_20251103_170226.csv'...
   - Carregando dados do ano 2024 de: 'relatorio_metricas_scores_UFCG_2024_20251103_170148.csv'...

✅ Relatório anual consolidado criado com 48 linhas.
                   Indicador/Métrica                  Tipo  \
0      ED1.1_Total_Emissions_KgCO2eq         Métrica Bruta   
1             ED1.1_Baseline_KgCO2eq              Baseline   
2         ED1.1_Variação_vs_Baseline  Variação vs Baseline   
3                        ED1.1_Score         Score (0 a 1)   
4  ED1.2_Avoidable_Emissions_KgCO2eq         Métrica Bruta   

                Valor   Ano  
0  270,626.40 KgCO2eq  2023  
1          285,440.32  2023  
2              -5.19%  2023  
3                 1.0  2023  
4      224.41 KgCO2eq  2023  


## 3. Preparar Dados para Visualização

In [3]:
if not df_full_report.empty:
    print("🔄 Preparando dados para os gráficos...")
    # 1. Filtra apenas Métricas Brutas
    df_metrics = df_full_report[df_full_report['Tipo'] == 'Métrica Bruta'].copy()
    
    # 2. Converte 'Valor' (formatado) para 'Valor_Numerico'
    df_metrics['Valor_Numerico'] = df_metrics['Valor'].apply(clean_numeric_value)
    
    # 3. Mapeia nomes longos para rótulos curtos para os gráficos
    mapa_rotulos = {
        'ED1.1_Total_Emissions_KgCO2eq': 'Emissões Totais (KgCO2e)',
        'ED2.1_Total_Costs_R$': 'Custo Total (R$)',
        'ED3.1_Total_Distance_Km': 'Distância Total (Km)',
        'ED2.3_Total_Trips': 'Total de Viagens',
        'ED1.4_Avg_Emissions_per_Trip': 'Emissão Média / Viagem',
        'ED2.4_Avg_Cost_per_Trip': 'Custo Médio / Viagem',
        'ED3.3_Avg_Distance_per_Trip': 'Distância Média / Viagem'
        # Adicione outros se desejar
    }
    df_metrics['Rótulo'] = df_metrics['Indicador/Métrica'].map(mapa_rotulos)
    
    # 4. Separa os DataFrames para os gráficos
    indicadores_totais = [
        'ED1.1_Total_Emissions_KgCO2eq',
        'ED2.1_Total_Costs_R$',
        'ED3.1_Total_Distance_Km',
        'ED2.3_Total_Trips'
    ]
    df_totais = df_metrics[df_metrics['Indicador/Métrica'].isin(indicadores_totais)]
    
    indicadores_medios = [
        'ED1.4_Avg_Emissions_per_Trip',
        'ED2.4_Avg_Cost_per_Trip',
        'ED3.3_Avg_Distance_per_Trip'
    ]
    df_medios = df_metrics[df_metrics['Indicador/Métrica'].isin(indicadores_medios)]
    
    print("✅ Dados prontos para visualização.")
    print("\nAmostra dos dados de Totais:")
    print(df_totais[['Ano', 'Rótulo', 'Valor_Numerico']].head())
else:
    print("⚠️ DataFrame consolidado vazio. Gráficos não podem ser gerados.")

🔄 Preparando dados para os gráficos...
✅ Dados prontos para visualização.

Amostra dos dados de Totais:
     Ano                    Rótulo  Valor_Numerico
0   2023  Emissões Totais (KgCO2e)       270626.40
7   2023          Custo Total (R$)      1551094.54
12  2023          Total de Viagens          320.00
15  2023      Distância Total (Km)      1395655.58
24  2024  Emissões Totais (KgCO2e)       270626.40


## 4. Criar Gráficos de Tendência (Totais Absolutos)

In [4]:
chart_totais = alt.Chart(pd.DataFrame()).mark_text(text="Nenhum dado total para exibir") # Default

if 'df_totais' in locals() and not df_totais.empty:
    print("🔄 Criando gráficos de Totais Absolutos (com ajuste de tamanho)...")
    
    # Gráfico base para os totais
    base_totais = alt.Chart(df_totais).mark_line(point=True).encode(
        # Eixo X: Ano (Ordinal para anos discretos)
        x=alt.X('Ano:O', axis=alt.Axis(title='Ano')),
        # Eixo Y: Valor Numérico
        y=alt.Y('Valor_Numerico:Q', axis=alt.Axis(title='Valor Total')),
        # Cor: Uma linha para cada métrica
        color=alt.Color('Rótulo:N', title='Métrica'),
        # Tooltip para interatividade
        tooltip=[
            alt.Tooltip('Ano:O'),
            alt.Tooltip('Rótulo:N', title='Métrica'),
            alt.Tooltip('Valor_Numerico:Q', title='Valor', format=',.0f') # Formato de inteiro
        ]
    )
    
    # Adiciona texto com o valor no gráfico
    text_totais = base_totais.mark_text(align='center', dy=-10, fontSize=10).encode(
        text=alt.Text('Valor_Numerico:Q', format=',.0f'),
        color=alt.value('black') # Cor do texto
    )
    
    # --- AJUSTE AQUI ---
    # Aplica .properties() ANTES do .facet() para definir o tamanho de CADA gráfico
    chart_totais = (base_totais + text_totais).properties(
        width=600,  # <-- Defina a largura de cada gráfico individual (ex: 600 pixels)
        height=150  # <-- Defina a altura de cada gráfico individual (ex: 150 pixels)
    ).facet(
        row=alt.Row('Rótulo:N', title='Métrica', header=alt.Header(labels=False, titleOrient="left")),
        # Faceta por linha, cada gráfico com seu próprio eixo Y
    ).resolve_scale(
        y='independent' # Cada gráfico (Emissão, Custo, etc.) tem sua própria escala Y
    ).properties(
         title='Tendência dos Totais Absolutos por Ano' # Título para o conjunto de gráficos
    )#.interactive()
    # --- FIM DO AJUSTE ---

# Exibe o gráfico no notebook (descomente se desejar)
# chart_totais
print("✅ Gráfico de Totais criado.")

🔄 Criando gráficos de Totais Absolutos (com ajuste de tamanho)...
✅ Gráfico de Totais criado.


## 5. Criar Gráficos de Tendência (Médias por Viagem)

In [5]:
chart_medios = alt.Chart(pd.DataFrame()).mark_text(text="Nenhum dado de médias para exibir") # Default

if 'df_medios' in locals() and not df_medios.empty:
    print("🔄 Criando gráficos de Médias por Viagem (com ajuste de tamanho)...")
    
    # Gráfico base para as médias
    base_medios = alt.Chart(df_medios).mark_line(point=True).encode(
        x=alt.X('Ano:O', axis=alt.Axis(title='Ano')),
        y=alt.Y('Valor_Numerico:Q', axis=alt.Axis(title='Valor Médio')),
        color=alt.Color('Rótulo:N', title='Métrica'),
        tooltip=[
            alt.Tooltip('Ano:O'),
            alt.Tooltip('Rótulo:N', title='Métrica'),
            alt.Tooltip('Valor_Numerico:Q', title='Valor', format=',.2f') # Formato com decimais
        ]
    )
    
    # Adiciona texto com o valor no gráfico
    text_medios = base_medios.mark_text(align='center', dy=-10, fontSize=10).encode(
        text=alt.Text('Valor_Numerico:Q', format=',.2f'),
        color=alt.value('black')
    )
    
    # --- AJUSTE AQUI ---
    # Aplica .properties() ANTES do .facet()
    chart_medios = (base_medios + text_medios).properties(
        width=600,  # <-- Use a mesma largura para consistência
        height=150  # <-- Use a mesma altura
    ).facet(
        row=alt.Row('Rótulo:N', title='Métrica', header=alt.Header(labels=False, titleOrient="left")),
    ).resolve_scale(
        y='independent' # Escalas Y independentes
    ).properties(
        title='Tendência das Médias por Viagem' # Título para o conjunto
    )#.interactive()
    # --- FIM DO AJUSTE ---

# Exibe o gráfico no notebook (descomente se desejar)
# chart_medios
print("✅ Gráfico de Médias criado.")

🔄 Criando gráficos de Médias por Viagem (com ajuste de tamanho)...
✅ Gráfico de Médias criado.


## 6. Montar e Salvar Dashboard de Tendência

In [6]:
if 'chart_totais' in locals() and 'chart_medios' in locals():
    print("🔄 Montando e salvando o dashboard de tendência...")
    
    # Combina os gráficos de Totais e Médias verticalmente
    dashboard_tendencia = alt.vconcat(
        chart_totais,
        chart_medios,
        title=f"Dashboard de Tendência Anual - {orgao}"
    ).resolve_scale(
        color='independent'
    )

    # Define o caminho de saída (na pasta base, não na pasta do ano)
    arquivo_dashboard = os.path.join(base_dir, f'dashboard_tendencia_anual_{orgao}.json')
    arquivo_dashboard_html = os.path.join(base_dir, f'dashboard_tendencia_anual_{orgao}.html')
    
    # Salva como JSON e HTML
    try:
        dashboard_tendencia.save(arquivo_dashboard)
        dashboard_tendencia.save(arquivo_dashboard_html)
        print(f"\n✅ Dashboard de Tendência salvo com sucesso em:")
        print(f"   - {arquivo_dashboard_html} (abrir no navegador)")
        print(f"   - {arquivo_dashboard} (abrir no VS Code com Altair Viewer)")
    except Exception as e_save_final:
         print(f"\n❌ Erro ao salvar dashboard final: {e_save_final}")
else:
    print("⚠️ Gráficos não foram criados, dashboard não foi salvo.")


🔄 Montando e salvando o dashboard de tendência...

✅ Dashboard de Tendência salvo com sucesso em:
   - dadosViagens\dashboard_tendencia_anual_UFCG.html (abrir no navegador)
   - dadosViagens\dashboard_tendencia_anual_UFCG.json (abrir no VS Code com Altair Viewer)


In [7]:
dashboard_tendencia

alt.VConcatChart(...)